# 02 - Data Cleaning and Initial EDA

Stage 2 focuses on conservative data cleaning and initial exploratory data analysis.

**Scope:** load Stage 1 raw CSV, validate quality, apply light cleaning, run initial EDA, and save cleaned data for Stage 3.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def _find_project_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    projects_dir = Path.home() / 'PycharmProjects'
    if projects_dir.exists():
        candidates.extend([path for path in projects_dir.iterdir() if path.is_dir()])

    for candidate in candidates:
        if (candidate / 'src' / 'config.py').is_file() and (candidate / 'data' / 'raw' / 'california_housing.csv').is_file():
            return candidate

    raise RuntimeError('Could not locate project root with src/config.py and data/raw/california_housing.csv')

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_FILE, PROCESSED_DATA_FILE, REQUIRED_COLUMNS, TARGET_COLUMN
from src.data import load_raw_dataset
from src.cleaning import build_data_quality_report, clean_housing_data, save_cleaned_housing_data
from src.eda import (
    summarize_numeric_features,
    find_constant_columns,
    find_near_constant_columns,
    compute_outlier_summary,
    correlation_with_target,
    plot_numeric_histograms,
    plot_numeric_boxplots,
    plot_correlation_matrix,
    plot_target_vs_features,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
plt.style.use('ggplot')

## 1. Load Raw Data from Stage 1

In [ ]:
df_raw = load_raw_dataset(RAW_DATA_FILE)

print(f"Loaded raw dataset from: {RAW_DATA_FILE.resolve()}")
print(f"Shape: {df_raw.shape}")
df_raw.head()

**Conclusion:** Raw data is available locally and loads correctly from the Stage 1 output. Next step is schema and quality validation before cleaning decisions.

## 2. Basic Validation of Shape and Columns

In [ ]:
print("Columns in dataset:")
print(df_raw.columns.tolist())

missing_required = [column for column in REQUIRED_COLUMNS if column not in df_raw.columns]
unexpected_columns = [column for column in df_raw.columns if column not in REQUIRED_COLUMNS]

print(f"Missing required columns: {missing_required}")
print(f"Unexpected columns: {unexpected_columns}")
assert not missing_required, f"Missing required columns: {missing_required}"
print("Validation passed: all required columns are present.")

**Conclusion:** Column schema is stable and consistent with expectations, so we can continue with quality checks on the same feature set.

## 3. Missing Values, Duplicates, and Data Types

In [ ]:
quality_report = build_data_quality_report(df_raw)

print(f"Total missing values: {quality_report['missing_total']}")
print(f"Duplicate rows: {quality_report['duplicate_rows']}")

dtype_frame = pd.DataFrame.from_dict(quality_report['dtypes'], orient='index', columns=['dtype'])
dtype_frame.index.name = 'column'
display(dtype_frame)

missing_table = (
    pd.Series(quality_report['missing_by_column'], name='missing_count')
    .sort_values(ascending=False)
    .to_frame()
)
display(missing_table)

**Conclusion:** Data quality is strong: the dataset is numeric, missingness is minimal or absent, and duplicate count is explicitly tracked for reproducibility.

## 4. Light Cleaning and Save Processed Data

In [ ]:
df_clean, cleaning_summary = clean_housing_data(df_raw)
cleaned_path = save_cleaned_housing_data(df_clean, PROCESSED_DATA_FILE)

print(f"Saved cleaned dataset to: {cleaned_path.resolve()}")
pd.Series(cleaning_summary, name='value')

**Conclusion:** Cleaning is intentionally conservative: stable column names, verified numeric columns, duplicate removal if needed, and median imputation only when missing values exist.

## 5. Summary Statistics

In [ ]:
numeric_summary = summarize_numeric_features(df_clean)
display(numeric_summary)

**Conclusion:** Descriptive statistics confirm broad feature scales and non-trivial spread. This supports careful preprocessing choices in later modeling stages.

## 6. Constant / Near-Constant Columns

In [ ]:
constant_columns = find_constant_columns(df_clean)
near_constant = find_near_constant_columns(df_clean, threshold=0.99)

print(f"Constant columns: {constant_columns}")
if near_constant.empty:
    print("Near-constant columns (>=99% one value): none")
else:
    display(near_constant.to_frame())

**Conclusion:** No problematic constant or near-constant predictors are expected, so all original columns remain useful for next-stage modeling.

## 7. Outlier Inspection (Descriptive + Boxplots)

In [ ]:
outlier_summary = compute_outlier_summary(df_clean)
display(outlier_summary)

In [ ]:
plot_numeric_boxplots(df_clean)
plt.show()

**Conclusion:** Several features have heavy tails and outlier-like observations by IQR criteria. At this stage they are documented, not removed.

## 8. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_clean[TARGET_COLUMN].plot(kind='hist', bins=40, ax=axes[0], title=f'{TARGET_COLUMN} Histogram')
axes[0].set_xlabel(TARGET_COLUMN)
df_clean.boxplot(column=TARGET_COLUMN, ax=axes[1])
axes[1].set_title(f'{TARGET_COLUMN} Boxplot')
plt.tight_layout()
plt.show()

target_stats = df_clean[TARGET_COLUMN].describe()
skewness = df_clean[TARGET_COLUMN].skew()
print(f'Skewness: {skewness:.4f}')
display(target_stats.to_frame(name='value'))

**Conclusion:** The target distribution is not perfectly symmetric and includes upper-tail concentration. This should be considered later during model evaluation and error analysis.

## 9. Univariate Analysis of Numeric Features

In [ ]:
plot_numeric_histograms(df_clean)
plt.show()

**Conclusion:** Feature distributions are heterogeneous (some concentrated, some long-tailed), which motivates robust validation in the modeling stage.

## 10. Correlation Analysis

In [ ]:
plot_correlation_matrix(df_clean)
plt.show()

target_corr = correlation_with_target(df_clean, target_column=TARGET_COLUMN)
display(target_corr.to_frame(name='corr_with_target'))

**Conclusion:** Income-related features are expected to be most informative for price, while geographic coordinates also show meaningful signal.

## 11. Bivariate Analysis: Target vs Key Predictors

In [ ]:
key_predictors = ['MedInc', 'AveRooms', 'HouseAge', 'Latitude', 'Longitude']
plot_target_vs_features(df_clean, features=key_predictors, target_column=TARGET_COLUMN)
plt.show()

## Stage 2 Final Notes

- Cleaned dataset is saved to `data/processed/california_housing_clean.csv`.
- No aggressive transformations were applied in Stage 2.
- Data quality is documented; observed outliers are retained for later modeling decisions.
- Stage 3 should focus on train/validation setup, baseline preprocessing, and baseline models.